In [14]:
import pandas as pd
import sqlite3

In [15]:
df=pd.read_csv('messy_sales_data.csv')

In [16]:
df.head()

,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep
0,1001,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma
1,1002,Priya Nair,NaN,Electronics,1.0,15000,2024-01-07,Delhi,Sunita Rao
2,1003,AMIT VERMA,Keyboard,Accessories,3.0,1200,2024-01-08,Bangalore,Anil Sharma
3,1004,Sunita Patel,Monitor,Electronics,NaN,22000,2024-01-10,Chennai,Ravi Kumar
4,1005,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       30 non-null     int64  
 1   customer_name  28 non-null     object 
 2   product        29 non-null     object 
 3   category       29 non-null     object 
 4   quantity       27 non-null     float64
 5   unit_price     30 non-null     int64  
 6   order_date     30 non-null     object 
 7   city           30 non-null     object 
 8   sales_rep      30 non-null     object 
dtypes: float64(1), int64(2), object(6)
memory usage: 2.2+ KB


In [18]:
df.isnull().sum()

,0
order_id,0
customer_name,2
product,1
category,1
quantity,3
unit_price,0
order_date,0
city,0
sales_rep,0


In [19]:
df.duplicated().sum()

np.int64(0)

In [20]:
df.dtypes

,0
order_id,int64
customer_name,object
product,object
category,object
quantity,float64
unit_price,int64
order_date,object
city,object
sales_rep,object


***TRANSFORM PHASE***

In [21]:
#removing duplicates

df=df.drop_duplicates()

print(df.duplicated().sum())

0


In [22]:
# Reload the original DataFrame to ensure correct data types from source
df = pd.read_csv('messy_sales_data.csv')

# Drop duplicates
df = df.drop_duplicates()

# Convert 'quantity' and 'unit_price' to numeric, coercing errors to NaN
# This handles cases where original CSV might contain non-numeric strings (e.g., 'Unknown')
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')

# Fill missing 'quantity' values (including those coerced from non-numeric strings) with 0
df['quantity'] = df['quantity'].fillna(0)

# Fill missing 'unit_price' values (including those coerced from non-numeric strings) with 0
df['unit_price'] = df['unit_price'].fillna(0)

# Fill missing values in other object-type columns with 'Unknown'
# (assuming 'order_date' is still object here, will be converted later)
object_cols = df.select_dtypes(include=['object']).columns
for col in object_cols:
    df[col] = df[col].fillna("Unknown")

df.isnull().sum()

,0
order_id,0
customer_name,0
product,0
category,0
quantity,0
unit_price,0
order_date,0
city,0
sales_rep,0


In [23]:
df['order_date']=pd.to_datetime(df['order_date'], format='mixed', dayfirst=True)

print(df['order_date'])

0    2024-01-05
1    2024-01-07
2    2024-01-08
3    2024-01-10
4    2024-01-05
5    2024-01-07
6    2024-01-12
7    2024-01-13
8    2024-01-15
9    2024-01-15
10   2024-01-18
11   2024-01-20
12   2024-01-22
13   2024-01-21
14   2024-01-25
15   2024-01-07
16   2024-01-28
17   2024-01-30
18   2024-02-01
19   2024-02-03
20   2024-02-05
21   2024-02-07
22   2024-01-15
23   2024-02-10
24   2024-02-12
25   2024-02-14
26   2024-02-15
27   2024-02-18
28   2024-02-20
29   2024-02-22
Name: order_date, dtype: datetime64[ns]


In [24]:
df['total_sales']=df['quantity'] * df['unit_price']

print(df['total_sales'])

0      90000.0
1      15000.0
2       3600.0
3          0.0
4      90000.0
5       8000.0
6       7000.0
7       2500.0
8      45000.0
9       6000.0
10     44000.0
11      4800.0
12         0.0
13     10500.0
14      5000.0
15      8000.0
16      6000.0
17     22000.0
18     45000.0
19      9000.0
20      7000.0
21      7500.0
22     45000.0
23      4800.0
24      4800.0
25      3500.0
26     44000.0
27    135000.0
28      6000.0
29         0.0
Name: total_sales, dtype: float64


In [25]:
df["city"] = df["city"].str.title()
df["category"] = df["category"].str.upper()

print(df[["city", "category"]])

                  city     category
0               Mumbai  ELECTRONICS
1                Delhi  ELECTRONICS
2            Bangalore  ACCESSORIES
3              Chennai  ELECTRONICS
4               Mumbai  ELECTRONICS
5                 Pune  ACCESSORIES
6            Hyderabad  ELECTRONICS
7               Mumbai  ACCESSORIES
8              Kolkata  ELECTRONICS
9              Chennai  ACCESSORIES
10               Delhi  ELECTRONICS
11           Bangalore  ACCESSORIES
12                Pune  ELECTRONICS
13               Kochi  ELECTRONICS
14               Surat  ACCESSORIES
15                Pune  ACCESSORIES
16           Hyderabad  ACCESSORIES
17               Delhi  ELECTRONICS
18           Hyderabad  ELECTRONICS
19             Lucknow  ACCESSORIES
20          Chandigarh  ELECTRONICS
21              Bhopal  ACCESSORIES
22             Kolkata  ELECTRONICS
23              Kanpur  ELECTRONICS
24             Kolkata  ACCESSORIES
25              Mumbai  ELECTRONICS
26               Kochi      

In [26]:
conn= sqlite3.connect('sales_database.db')

df.to_sql(name='sales', con=conn, if_exists='replace', index=False)

30

In [27]:
query="SELECT * FROM sales LIMIT 5"

pd.read_sql(query, conn)

,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep,total_sales
0,1001,Ramesh Kumar,Laptop,ELECTRONICS,2.0,45000,2024-01-05 00:00:00,Mumbai,Anil Sharma,90000.0
1,1002,Priya Nair,Unknown,ELECTRONICS,1.0,15000,2024-01-07 00:00:00,Delhi,Sunita Rao,15000.0
2,1003,AMIT VERMA,Keyboard,ACCESSORIES,3.0,1200,2024-01-08 00:00:00,Bangalore,Anil Sharma,3600.0
3,1004,Sunita Patel,Monitor,ELECTRONICS,0.0,22000,2024-01-10 00:00:00,Chennai,Ravi Kumar,0.0
4,1005,Ramesh Kumar,Laptop,ELECTRONICS,2.0,45000,2024-01-05 00:00:00,Mumbai,Anil Sharma,90000.0


In [28]:
query2="""
SELECT SUM(total_sales) AS total_revenue
FROM sales
"""

pd.read_sql(query2, conn)

,total_revenue
0,679000.0


POST CLEANING VALIDATION REPORT


In [29]:
#check remaining null values

print("NULL VALUES CHECK")
print("-" * 40)

null_values = df.isnull().sum()

print(null_values)

NULL VALUES CHECK
----------------------------------------
order_id         0
customer_name    0
product          0
category         0
quantity         0
unit_price       0
order_date       0
city             0
sales_rep        0
total_sales      0
dtype: int64


In [30]:
#checking duplicate records
print("DUPLICATE RECORDS CHECK")
print("-" * 40)

duplicates = df.duplicated().sum()

print(f"Total Duplicate Rows: {duplicates}")

DUPLICATE RECORDS CHECK
----------------------------------------
Total Duplicate Rows: 0


In [31]:
#check dataset shape

print("DATASET SHAPE")
print("-" * 40)

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

DATASET SHAPE
----------------------------------------
Rows: 30
Columns: 10


In [32]:
#check datatypes

print("DATA TYPES CHECK")
print("-" * 40)

print(df.dtypes)

DATA TYPES CHECK
----------------------------------------
order_id                  int64
customer_name            object
product                  object
category                 object
quantity                float64
unit_price                int64
order_date       datetime64[ns]
city                     object
sales_rep                object
total_sales             float64
dtype: object


In [33]:
#check negative or invalid quantities

print("INVALID QUANTITY CHECK")
print("-" * 40)

invalid_quantity = df[df["quantity"] <= 0]

print(f"Invalid Quantity Rows: {len(invalid_quantity)}")

invalid_quantity.head()

INVALID QUANTITY CHECK
----------------------------------------
Invalid Quantity Rows: 3


,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep,total_sales
3,1004,Sunita Patel,Monitor,ELECTRONICS,0.0,22000,2024-01-10,Chennai,Ravi Kumar,0.0
12,1013,Meera Joshi,Laptop,ELECTRONICS,0.0,45000,2024-01-22,Pune,Ravi Kumar,0.0
29,1030,Kavya Nambiar,Webcam,ACCESSORIES,0.0,2500,2024-02-22,Thrissur,Anil Sharma,0.0


In [34]:
#check negative prices

print("INVALID QUANTITY CHECK")
print("-" * 40)

invalid_quantity = df[df["quantity"] <= 0]

print(f"Invalid Quantity Rows: {len(invalid_quantity)}")

invalid_quantity.head()

INVALID QUANTITY CHECK
----------------------------------------
Invalid Quantity Rows: 3


,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep,total_sales
3,1004,Sunita Patel,Monitor,ELECTRONICS,0.0,22000,2024-01-10,Chennai,Ravi Kumar,0.0
12,1013,Meera Joshi,Laptop,ELECTRONICS,0.0,45000,2024-01-22,Pune,Ravi Kumar,0.0
29,1030,Kavya Nambiar,Webcam,ACCESSORIES,0.0,2500,2024-02-22,Thrissur,Anil Sharma,0.0


In [35]:
#validating date values

print("ORDER DATE VALIDATION")
print("-" * 40)

print(df["order_date"].dtype)


ORDER DATE VALIDATION
----------------------------------------
datetime64[ns]


In [36]:
#validate total sales column

print("TOTAL SALES VALIDATION")
print("-" * 40)

df["calculated_sales"] = df["quantity"] * df["unit_price"]

validation = (
    df["total_sales"] == df["calculated_sales"]
).all()

print(f"Total Sales Correctly Calculated: {validation}")

TOTAL SALES VALIDATION
----------------------------------------
Total Sales Correctly Calculated: True


In [37]:
#check unique categoriess

print("UNIQUE CATEGORIES")
print("-" * 40)

print(df["category"].unique())

UNIQUE CATEGORIES
----------------------------------------
['ELECTRONICS' 'ACCESSORIES' 'UNKNOWN']


In [38]:
#check unique cities

print("UNIQUE CITIES")
print("-" * 40)

print(df["city"].unique())

UNIQUE CITIES
----------------------------------------
['Mumbai' 'Delhi' 'Bangalore' 'Chennai' 'Pune' 'Hyderabad' 'Kolkata'
 'Kochi' 'Surat' 'Lucknow' 'Chandigarh' 'Bhopal' 'Kanpur'
 'Thiruvananthapuram' 'Allahabad' 'Thrissur']


In [39]:
#FINAL VALIDATION SUMMARY REPORT

print("=" * 50)
print("POST CLEANING VALIDATION REPORT")
print("=" * 50)

print(f"Total Rows: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")

print("\nNull Values Remaining:")
print(df.isnull().sum().sum())

print("\nDuplicate Rows Remaining:")
print(df.duplicated().sum())

print("\nInvalid Quantity Rows:")
print(len(df[df["quantity"] <= 0]))

print("\nInvalid Unit Price Rows:")
print(len(df[df["unit_price"] <= 0]))

print("\nOrder Date Data Type:")
print(df["order_date"].dtype)

print("\nValidation Completed Successfully")
print("=" * 50)

POST CLEANING VALIDATION REPORT
Total Rows: 30
Total Columns: 11

Null Values Remaining:
0

Duplicate Rows Remaining:
0

Invalid Quantity Rows:
3

Invalid Unit Price Rows:
0

Order Date Data Type:
datetime64[ns]

Validation Completed Successfully


In [40]:
#total revenue generated

total_revenue = df["total_sales"].sum()

print("TOTAL REVENUE")
print("-" * 30)

print(f"₹ {total_revenue:,.2f}")

TOTAL REVENUE
------------------------------
₹ 679,000.00


In [41]:
#top 5 selling products

top_products = (
    df.groupby("product")["quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
)

print("TOP 5 SELLING PRODUCTS")
print("-" * 30)

print(top_products)

TOP 5 SELLING PRODUCTS
------------------------------
      product  quantity
0     USB Hub      33.0
1       Mouse      26.0
2    Keyboard      17.0
3      Laptop      10.0
4  Headphones       8.0


In [42]:
#revenue by category

category_revenue = (
    df.groupby("category")["total_sales"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

print("CATEGORY WISE REVENUE")
print("-" * 30)

print(category_revenue)

CATEGORY WISE REVENUE
------------------------------
      category  total_sales
0  ELECTRONICS     563800.0
1  ACCESSORIES      71200.0
2      UNKNOWN      44000.0


In [43]:
#revenue by city

city_revenue = (
    df.groupby("city")["total_sales"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

print("CITY WISE REVENUE")
print("-" * 30)

print(city_revenue)

CITY WISE REVENUE
------------------------------
                  city  total_sales
0               Mumbai     186000.0
1   Thiruvananthapuram     135000.0
2              Kolkata      94800.0
3                Delhi      81000.0
4            Hyderabad      58000.0
5                Kochi      54500.0
6                 Pune      16000.0
7              Lucknow       9000.0
8            Bangalore       8400.0
9               Bhopal       7500.0
10          Chandigarh       7000.0
11           Allahabad       6000.0
12             Chennai       6000.0
13               Surat       5000.0
14              Kanpur       4800.0
15            Thrissur          0.0


In [44]:
#product summary table

product_summary = (
    df.groupby("product")
    .agg({
        "quantity": "sum",
        "total_sales": "sum",
        "unit_price": "mean"
    })
    .reset_index()
)

product_summary.columns = [
    "product",
    "total_quantity_sold",
    "total_revenue",
    "average_price"
]

print("PRODUCT SUMMARY TABLE")
print("-" * 30)

print(product_summary)

PRODUCT SUMMARY TABLE
------------------------------
      product  total_quantity_sold  total_revenue  average_price
0  Headphones                  8.0        28000.0         3500.0
1    Keyboard                 17.0        20400.0         1200.0
2      Laptop                 10.0       450000.0        45000.0
3     Monitor                  5.0       110000.0        22000.0
4       Mouse                 26.0        20800.0          800.0
5     USB Hub                 33.0        19800.0          600.0
6     Unknown                  1.0        15000.0        15000.0
7      Webcam                  6.0        15000.0         2500.0


In [47]:
product_summary.to_csv("product_summary.csv", index=False)

print("Summary tables exported successfully")

Summary tables exported successfully


In [51]:
import requests

API_KEY = "b0e3bcfae63981192b109935d1acadb9"

weather_data = []

for city in df["city"].unique():

    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}&units=metric"

    response = requests.get(url)

    data = response.json()

    # Check if the API call was successful before processing data
    if response.status_code == 200:
        weather_data.append({
            "city": city,
            "temperature": data["main"]["temp"],
            "weather": data["weather"][0]["main"]
        })
    else:
        print(f"Error fetching weather for {city}: {data.get('message', 'Unknown error')}")
        weather_data.append({
            "city": city,
            "temperature": None,
            "weather": "Error"
        })

weather_df = pd.DataFrame(weather_data)

weather_df.head()

Error fetching weather for Mumbai: Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.
Error fetching weather for Delhi: Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.
Error fetching weather for Bangalore: Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.
Error fetching weather for Chennai: Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.
Error fetching weather for Pune: Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.
Error fetching weather for Hyderabad: Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.
Error fetching weather for Kolkata: Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.
Error fetching weather for Kochi: Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.
Error fetching weather for Surat: Invalid API key. P

,city,temperature,weather
0,Mumbai,None,Error
1,Delhi,None,Error
2,Bangalore,None,Error
3,Chennai,None,Error
4,Pune,None,Error
